# 06. Transformer Forward Pass — text から logits まで

## 学習目標

- 実文を入力し、**全中間 tensor の shape と実値**を追跡する（spec §8.4）
- residual stream が層を通じてどう成長するかを測る
- logits → 次トークン分布 → 生成、の最後の1マイルを繋ぐ

## 追跡する形状（B=バッチ, T=系列長, D=隠れ次元, H=head数, V=語彙）

```text
input_ids              [B, T]
token_embeddings       [B, T, D]
Q, K, V                [B, H, T, D/H]
attention_scores       [B, H, T, T]
attention_weights      [B, H, T, T]
attention_output       [B, T, D]
mlp_output             [B, T, D]
residual_stream        [B, T, D]
logits                 [B, T, V]
probabilities          [B, T, V]
```

In [1]:
# 共通セットアップ（全ノートブック同一）
import warnings
warnings.filterwarnings("ignore")

import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT={ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}")

ROOT=/home/kazumasa/projects/jp_llm_lab
device=cuda, torch=2.11.0+cu128


In [2]:
from jp_llm_lab.models.config import ModelConfig
from jp_llm_lab.models.transformer import ClassicalGPT
from jp_llm_lab.training.trainer import load_checkpoint
from jp_llm_lab.tokenization.char_tokenizer import CharTokenizer

RUN = ROOT / "experiments/runs/m1_model_s_smoke_seed42"
assert RUN.exists(), "先に `make -C jp_llm_lab train-smoke` を実行してください"
tok = CharTokenizer.load(RUN / "tokenizer.json")
ckpt = sorted(RUN.glob("checkpoints/ckpt_100pct_*.pt"))[0]
payload = torch.load(ckpt, map_location="cpu", weights_only=False)
model = ClassicalGPT(ModelConfig.from_dict(payload["model_cfg"]))
load_checkpoint(ckpt, model)
model.eval()
print(f"loaded: {ckpt.name} (step {payload['step']}, {payload['tokens_seen']:,} tokens 学習済み)")

loaded: ckpt_100pct_step000400.pt (step 400, 6,553,600 tokens 学習済み)


In [3]:
from jp_llm_lab.visualization.params_viz import param_breakdown_figure
fig = param_breakdown_figure(model.param_breakdown())
fig.show()

In [4]:
# 実文の forward pass を trace（学習と同一のコードパスに dict を通すだけ — 二重実装なし）
s = "私はその人を常に先生と呼んでいた。"
ids = torch.tensor([tok.encode(s)])
trace = model.trace_forward(ids)

rows = [(name, str(tuple(t.shape)), str(t.dtype).replace("torch.", "")) for name, t in trace.items()]
pd.DataFrame(rows, columns=["tensor", "shape", "dtype"])

,tensor,shape,dtype
0,input_ids,"(1, 17)",int64
1,token_embeddings,"(1, 17, 128)",float32
2,position_embeddings,"(1, 17, 128)",float32
3,embeddings,"(1, 17, 128)",float32
4,block0.attn.q,"(1, 4, 17, 32)",float32
5,block0.attn.k,"(1, 4, 17, 32)",float32
6,block0.attn.v,"(1, 4, 17, 32)",float32
7,block0.attn.scores,"(1, 4, 17, 17)",float32
8,block0.attn.attn_weights,"(1, 4, 17, 17)",float32
9,block0.attn.out,"(1, 17, 128)",float32


In [5]:
# debug mode: shape だけでなく実値の先頭を覗く
def peek(name, k=4):
    t = trace[name].float()
    vals = [round(float(v), 3) for v in t.flatten()[:k]]
    print(f"{name:28s} {str(tuple(t.shape)):>16s}  first{k}: {vals}")

for name in ["input_ids", "token_embeddings", "embeddings",
             "block0.attn.q", "block0.attn.scores", "block0.attn.attn_weights",
             "block0.resid_after_mlp", "logits", "probabilities"]:
    peek(name)

input_ids                             (1, 17)  first4: [1382.0, 59.0, 41.0, 58.0]
token_embeddings                 (1, 17, 128)  first4: [-0.028, -0.002, 0.032, -0.088]
embeddings                       (1, 17, 128)  first4: [0.036, 0.023, 0.04, -0.131]
block0.attn.q                  (1, 4, 17, 32)  first4: [-0.706, -0.408, 0.45, 0.234]
block0.attn.scores             (1, 4, 17, 17)  first4: [-0.54, -inf, -inf, -inf]
block0.attn.attn_weights       (1, 4, 17, 17)  first4: [1.0, 0.0, 0.0, 0.0]
block0.resid_after_mlp           (1, 17, 128)  first4: [-0.487, 0.258, -0.984, 0.29]
logits                          (1, 17, 2068)  first4: [-7.154, -7.161, -7.233, -7.211]
probabilities                   (1, 17, 2068)  first4: [0.0, 0.0, 0.0, 0.0]


In [6]:
# residual stream の RMS 成長と、各 branch の書き込み量 r_l = ‖Δh‖/‖h‖
from jp_llm_lab.instrumentation.activation_stats import residual_update_ratios

names = ["embeddings"] + [f"block{i}.resid_after_mlp" for i in range(4)] + ["final_norm"]
rms = [float(trace[n].float().pow(2).mean().sqrt()) for n in names]
fig = go.Figure(go.Scatter(x=[n.replace(".resid_after_mlp", "") for n in names], y=rms,
                           mode="lines+markers", line=dict(color="#1f77b4")))
fig.update_layout(title="residual stream RMS（層を経るごとに蓄積で成長）",
                  yaxis_title="RMS", template="plotly_white", height=380)
fig.show()

ratios = residual_update_ratios(trace, n_layers=4)
fig2 = go.Figure(go.Bar(x=list(ratios), y=list(ratios.values()), marker_color="#ff7f0e"))
fig2.update_layout(title="branch別 residual 書き込み比 r_l = ‖Δh‖/‖h‖（この1文に対して）",
                   yaxis_title="r_l", template="plotly_white", height=380)
fig2.show()

In [7]:
# 最終位置の logits → 次トークン分布 top10
probs = trace["probabilities"][0, -1]
top = torch.topk(probs, 10)
chars_top = [tok.id_to_token(int(i)).replace("\n", "⏎") for i in top.indices]
entropy = float(-(probs * torch.log(probs.clamp(min=1e-12))).sum())
fig = go.Figure(go.Bar(x=chars_top, y=[float(v) for v in top.values], marker_color="#2ca02c"))
fig.update_layout(title=f"「{s}」の次トークン分布 top10（エントロピー {entropy:.2f} nats）",
                  yaxis_title="確率", template="plotly_white", height=380)
fig.show()
print(f"原文の続きは「だ」（…呼んでいた。だから…）→ モデルのP(だ) = {float(probs[tok.token_to_id('だ')]):.3f}")

原文の続きは「だ」（…呼んでいた。だから…）→ モデルのP(だ) = 0.004


In [8]:
# 生成: greedy vs temperature 0.7（同じモデル・同じprompt）
from jp_llm_lab.generation.sampler import SamplingConfig, generate

ids0 = torch.tensor([tok.encode("私は")])
out_g, _ = generate(model, ids0, SamplingConfig(max_new_tokens=80, greedy=True))
out_s, _ = generate(model, ids0, SamplingConfig(max_new_tokens=80, temperature=0.7, seed=0))
print("greedy :", tok.decode(out_g[0].tolist()).replace("\n", "⏎"))
print()
print("temp0.7:", tok.decode(out_s[0].tolist()).replace("\n", "⏎"))

greedy : 私はその時、私の時、私はその時の時には私の事を聞いていた。⏎　私はその時の時にはその時についてその時に先生の言葉を見ていた。⏎「どうしてもう」⏎「おっともう」と先生

temp0.7: 私はその口でなくなった。⏎⏎⏎「しかし先生は私が先生の宅へ出ているべきでしょう。何しからますかしそれどもそのまだけは書いという真情をしてもしました。しかしそれはその


## Observation / Interpretation / Caveat

- **Observation**: shape 表は spec §8.4 の理論形状と完全一致。residual RMS は層とともに単調増加
  （pre-LN 構成の既知挙動）。冒頭文の次トークン分布は「⏎（改行）0.28・私 0.14・先 0.10…」と
  **句点後の一般的統計**に集中し、原文の実際の続き「だ」は P=0.004 で top10 圏外 —
  この規模のモデルは学習データ冒頭文すら逐語再現しない。
- **Interpretation**: 「Transformer は residual stream に各 block が読み書きするパイプライン」
  という描像が、RMS 成長と r_l でそのまま数値化できる。greedy の反復と temperature の
  多様性のトレードオフは、同じ logits に対する**決定則の違い**にすぎない。
- **Caveat**: r_l や RMS は**この1文**に対する値。分布としての統計は NB16（M2）で
  学習中の全評価バッチに対して追跡する。生成品質の evaluation は定性であり、
  repetition率などの定量化は M5。

## 確認問題

1. `attention_scores` の shape が $[B,H,T,T]$ になる理由を $Q,K$ の shape から導け。
2. weight tying により `lm_head` の shape はどの tensor と一致するか。
3. residual stream の RMS が層とともに増えるのは、pre-LN のどの性質によるか。

**次へ（M2）**: NB02 コーパス探索、NB07 アーキテクチャ可視化、NB13+ 学習ダイナミクス。